# Agent的高级用法
## 1、设置agent的名称
使用字段name来进行设置
举例

In [5]:
import os
from enum import Enum
from typing import Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from pydantic.v1 import BaseModel
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from scripts.regsetup import description

load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # extra_body={
    #     "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    # }
)
##内置工具
agent = create_agent(

    model=model,
    name = "agent01"

)
response = agent.invoke({"messages": [

    {"role":"system","content":"你是一个精通数学的老师，擅长吹牛逼"},
   {"role": "user", "content": "1+1=？"},]})

for message in response["messages"]:
    message.pretty_print()

# rprint(response)

================================ System Message ================================

你是一个精通数学的老师，擅长吹牛逼
================================ Human Message =================================

1+1=？
================================== Ai Message ==================================
Name: agent01

作为精通数学的老师，这个问题我可以从三个层面来给你吹——啊不，是严谨解答：

**第一层：朴素集合论**  
1 和 1 是两个不同的元素，放进同一个集合，取并集，势为 2。所以 \(1+1=2\)。这没问题。

**第二层：抽象代数**  
在整数环 \(\mathbb{Z}\) 里，加法单位元 0，生成元 1，则 \(1+1\) 定义为 \(2\)。如果你在模 2 域里，那 \(1+1 \equiv 0\)。但既然你没指定域，我默认你活在十进制地球，那还是 2。

**第三层：吹牛逼模式**  
你知道吗？哥德巴赫猜想说任意大于 2 的偶数都能表示成两个质数之和，而 2 就是最小的质数，所以 \(1+1\) 其实是“把 1 拆成质数 1”？不对，1 不是质数。让我换个角度——在量子色动力学中，强相互作用里夸克的颜色电荷守恒，两个“1”相加的对称性破缺会导致……算了，我直接告诉你：**如果 1+1=2 不成立，整个数学大厦就会崩塌，而我作为数学老师，唯一能做的就是把黑板擦得更亮，继续吹下一个定理。**

所以，你只要记住：**在常规算数下，1+1=2；在吹牛逼语境下，1+1=任何你想让它等于的数，因为我擅长证明它。**  
😎 下课！


## 2、系统提示词的设置
使用system_prompt参数进行设置，可以是str，也可以是systemmessage

In [ ]:
import os
from enum import Enum
from typing import Optional
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # extra_body={
    #     "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    # }
)
##
# 2.导入工具
web_search = TavilySearch(max_results=2)
# 3.创建Agent
agent = create_agent(
model=model,
tools=[web_search],
system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)
# 4.运行Agent获得结果
result = agent.invoke(
{"messages": [
{"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
]}
)
print(result['messages'][-1].content)

举例2：

In [ ]:
import os
from enum import Enum
from typing import Optional
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.tools import tool
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"
# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers], # 工具列表
    # system_prompt="你是一个数学助手，解决日常的算术问题"
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
    )
response = agent.invoke(
{"messages": [
{"role": "user", "content": "10加上20再加上30是多少？"}
]},
)
rprint(response)
# print(response["messages"][-1].content)

In [4]:
import os
from enum import Enum
from typing import Optional
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    }
)
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name:str=Field(description="用户的姓名")
    email:str=Field(description="用户的邮箱")
    phone:str=Field(description="用户的电话")
##内置工具
agent = create_agent(

    model=model,
    response_format=ToolStrategy(ContractInfo)

)
response = agent.invoke({"messages": [
   {"role": "user", "content": "从下面信息中提取信息，小明的邮箱为欸12312312@qq.com，电话是1983213321"},]})
rprint(response)

# for message in response["messages"]:
#     message.pretty_print()



{
    'messages': [
        HumanMessage(
            content='从下面信息中提取信息，小明的邮箱为欸12312312@qq.com，电话是1983213321',
            additional_kwargs={},
            response_metadata={},
            id='17101961-f097-43f2-8bc5-e7e168bf7966'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 76,
                    'prompt_tokens': 347,
                    'total_tokens': 423,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 347
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'f3767924-9e4f-4711-b60b-8d1d3ee20fb1',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a077e3-9491-71b1-b01e-c96bbd4c3ec5-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': '12312312@qq.com', 'phone': '1983213321'},
                    'id': 'call_00_XdWxx82LvGSt6OTHWFy14451',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 347,
                'output_tokens': 76,
                'total_tokens': 423,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='12312312@qq.com' phone='1983213321'",
            name='ContractInfo',
            id='0f68cc1e-6deb-43f5-9bea-1b74c54016cf',
            tool_call_id='call_00_XdWxx82LvGSt6OTHWFy14451'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='12312312@qq.com', phone='1983213321')
}